# XAUUSD PPO — tensor-optix Edition
Autonomous training loop via [tensor-optix](https://github.com/sup3rus3r/tensor-optix).  
Same model, environment and reward as `training_notebook(4)`.  
tensor-optix owns: evaluation · checkpointing · PBT hyperparameter tuning · policy evolution · ensemble management.

using dataset https://www.kaggle.com/datasets/mohsiniqbalgoni/gold-xauusd-1min-2004-2021

In [ ]:

import tensorflow
import tensorflow as tf
import numpy as np
import pandas as pd
import os, json, logging, time, math
from datetime import datetime
from collections import deque

# tensor-optix
from tensor_optix import RLOptimizer
from tensor_optix.core.base_agent import BaseAgent
from tensor_optix.core.base_evaluator import BaseEvaluator
from tensor_optix.core.types import EpisodeData, EvalMetrics, HyperparamSet
from tensor_optix.pipeline.batch_pipeline import BatchPipeline
from tensor_optix.core.policy_manager import PolicyManager
from tensor_optix.core.checkpoint_registry import CheckpointRegistry
from tensor_optix.core.meta_controller import MetaController
from tensor_optix.core.regime_detector import RegimeDetector
from tensor_optix.core.ensemble_agent import EnsembleAgent
from tensor_optix.core.loop_controller import LoopCallback
from tensor_optix.optimizers.pbt_optimizer import PBTOptimizer

print("TF:", tf.__version__)


In [ ]:

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices('GPU')
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)
print(f"GPUs: {[g.name for g in gpus] or 'CPU only'}")

os.makedirs("output/logs",        exist_ok=True)
os.makedirs("output/checkpoints", exist_ok=True)
os.makedirs("output/to_checkpoints", exist_ok=True)

LOG_PATH   = "output/logs/to_training.log"
JSONL_PATH = "output/logs/to_training.jsonl"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        logging.FileHandler(LOG_PATH, mode="w"),
        logging.StreamHandler(),
    ],
)
log = logging.getLogger("xauusd_to")

def jlog(obj):
    with open(JSONL_PATH, "a") as f:
        f.write(json.dumps({**obj, "ts": datetime.utcnow().isoformat()}) + "\n")

def _f32(x):
    if isinstance(x, np.ndarray): return x.astype(np.float32)
    try: return tf.cast(x, tf.float32)
    except Exception: return x


In [ ]:

CFG = dict(
    # Data selection
    N_STEPS_SELECT  = 200_000,
    SCAN_STEP       = 500,
    N_TRAIN_WINDOWS = 20,
    TRAIN_FRAC      = 0.80,
    MAX_START_ROW   = 160_000,   # 80% of N_STEPS_SELECT — no val leakage

    # State
    N_MARKET_FEAT   = 9,
    N_POS_FEAT      = 4,

    # Execution costs (Cont & Kukanov 2012)
    SPREAD_PRICE    = 0.09,
    COMMISSION_PRICE= 0.035,
    LOT_SIZE        = 0.10,
    INIT_EQUITY     = 10_000.0,

    # Reward
    LAM_DD          = 2.0,
    REWARD_CLIP     = 3.0,
    FLAT_PENALTY    = 0.001,

    # PPO — initial values; PBT will auto-tune entropy_coeff & clip_ratio
    LR_START        = 3e-4,
    LR_END          = 1e-5,
    CLIP_EPS        = 0.20,
    GAMMA           = 0.99,
    GAE_LAMBDA      = 0.95,
    ENTROPY_COEFF   = 0.06,
    ENTROPY_TARGET  = 0.45,
    VALUE_COEFF     = 0.5,
    MAX_GRAD_NORM   = 0.5,
    N_ROLLOUT_STEPS = 2048,
    N_EPOCHS        = 4,
    TARGET_KL       = 0.015,
    MINI_BATCH_SIZE = 256,
)

STATE_DIM = CFG['N_MARKET_FEAT'] + CFG['N_POS_FEAT']   # 13
N_ACTIONS = 3   # 0=FLAT, 1=LONG, 2=SHORT

log.info(f"CFG loaded | state_dim={STATE_DIM} | rollout={CFG['N_ROLLOUT_STEPS']}")


In [ ]:

log.info("Loading dataset.csv ...")
t0  = time.time()
raw = pd.read_csv("dataset.csv", sep=";", parse_dates=["date"], dayfirst=False)
raw.sort_values("date", inplace=True)
raw.reset_index(drop=True, inplace=True)
log.info(f"Loaded {len(raw):,} rows in {time.time()-t0:.1f}s | "
         f"Date range: {raw['date'].iloc[0]} → {raw['date'].iloc[-1]} | "
         f"Close range: [{raw['close'].min():.2f}, {raw['close'].max():.2f}]")


In [ ]:

log.info("Computing log-returns for balance scoring ...")
raw['log_ret'] = np.log(raw['close'] / raw['close'].shift(1))
raw.dropna(inplace=True)
raw.reset_index(drop=True, inplace=True)

N     = CFG['N_STEPS_SELECT']
STEP  = CFG['SCAN_STEP']
total = len(raw)
train_cutoff = int(total * CFG['TRAIN_FRAC'])

ret_arr = raw['log_ret'].values
vol_arr = raw['tick_volume'].values.astype(np.float64)

def _scan_windows(starts, ret_arr, vol_arr, N):
    balances  = np.empty(len(starts), dtype=np.int64)
    vol_means = np.empty(len(starts), dtype=np.float64)
    for i, s in enumerate(starts):
        chunk_ret    = ret_arr[s: s + N]
        chunk_vol    = vol_arr[s: s + N]
        up           = int(np.sum(chunk_ret > 0))
        dn           = int(np.sum(chunk_ret < 0))
        balances[i]  = abs(up - dn)
        vol_means[i] = chunk_vol.mean()
    return balances, vol_means

def _pick_k_stratified(starts, balances, vol_means, N, K):
    """Divide start-index range into K equal bands; pick most balanced per band."""
    band_size = max(1, len(starts) // K)
    selected  = []
    for k in range(K):
        lo = k * band_size
        hi = lo + band_size if k < K - 1 else len(starts)
        band_starts = starts[lo:hi]
        band_bal    = balances[lo:hi]
        best        = int(band_starts[np.argmin(band_bal)])
        selected.append(best)
    return selected

K            = CFG['N_TRAIN_WINDOWS']
train_starts = np.arange(0, train_cutoff - N, STEP)
log.info(f"Scanning {len(train_starts):,} train windows (stride={STEP}) ...")
t1 = time.time()
train_bal, train_vol = _scan_windows(train_starts, ret_arr, vol_arr, N)
log.info(f"Train scan done in {time.time()-t1:.1f}s | Balance range: {train_bal.min()} → {train_bal.max()}")

best_train_starts = _pick_k_stratified(train_starts, train_bal, train_vol, N, K)

train_slices = []
for i, s in enumerate(best_train_starts):
    sl = raw.iloc[s: s + N].copy().reset_index(drop=True)
    train_slices.append(sl)
    log.info(f"  Train[{i}]: idx={s} | {sl['date'].iloc[0]} → {sl['date'].iloc[-1]} | "
             f"close=[{sl['close'].min():.0f},{sl['close'].max():.0f}]")

# Val window: best from last 20% of dataset
val_starts = np.arange(train_cutoff, total - N, STEP)
val_bal, val_vol = _scan_windows(val_starts, ret_arr, vol_arr, N)
best_val = _pick_k_stratified(val_starts, val_bal, val_vol, N, 1)[0]
val_slice = raw.iloc[best_val: best_val + N].copy().reset_index(drop=True)
log.info(f"Val window: idx={best_val} | {val_slice['date'].iloc[0]} → {val_slice['date'].iloc[-1]} | "
         f"close=[{val_slice['close'].min():.0f},{val_slice['close'].max():.0f}]")
slice_df = train_slices[0]


In [ ]:

def featurize(sl: pd.DataFrame):
    """9 features: lr1, lr5, hl, rsi14, atr_n, vol_z, macd_h, hr_sin, hr_cos"""
    df = sl.copy()
    df['lr1']  = np.log(df['close'] / df['close'].shift(1)).fillna(0)
    df['lr5']  = np.log(df['close'] / df['close'].shift(5)).fillna(0)
    df['hl']   = ((df['high'] - df['low']) / (df['close'] + 1e-8)).fillna(0)

    # RSI-14
    delta = df['close'].diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    rs    = gain / (loss + 1e-8)
    df['rsi14'] = (100 - 100 / (1 + rs)).fillna(50) / 100.0

    # ATR-14 (normalised)
    tr  = pd.concat([df['high'] - df['low'],
                     (df['high'] - df['close'].shift()).abs(),
                     (df['low']  - df['close'].shift()).abs()], axis=1).max(axis=1)
    atr = tr.rolling(14).mean().fillna(tr)
    df['atr_n'] = (atr / (df['close'] + 1e-8)).fillna(0)

    # Volume z-score
    vm = df['tick_volume'].rolling(20).mean().fillna(df['tick_volume'])
    vs = df['tick_volume'].rolling(20).std().fillna(1.0)
    df['vol_z'] = ((df['tick_volume'] - vm) / (vs + 1e-8)).clip(-3, 3).fillna(0)

    # MACD histogram (normalised by ATR)
    ema12 = df['close'].ewm(span=12, adjust=False).mean()
    ema26 = df['close'].ewm(span=26, adjust=False).mean()
    macd  = ema12 - ema26
    signal= macd.ewm(span=9, adjust=False).mean()
    df['macd_h'] = ((macd - signal) / (atr + 1e-8)).clip(-3, 3).fillna(0)

    # Hour-of-day cyclical encoding
    hour = df['date'].dt.hour + df['date'].dt.minute / 60.0
    df['hr_sin'] = np.sin(2 * np.pi * hour / 24)
    df['hr_cos'] = np.cos(2 * np.pi * hour / 24)

    feat_names = ['lr1','lr5','hl','rsi14','atr_n','vol_z','macd_h','hr_sin','hr_cos']
    feat = df[feat_names].values.astype(np.float32)
    prices = df['close'].values.astype(np.float32)
    atrs   = atr.values.astype(np.float32)
    return feat, prices, atrs, feat_names

log.info(f"Engineering features for {K} train + 1 val window ...")
train_window_pool = []
for i, sl in enumerate(train_slices):
    feat, prices, atrs, feat_names = featurize(sl)
    assert not np.any(np.isnan(feat)) and not np.any(np.isinf(feat))
    log.info(f"  Train[{i}]: shape={feat.shape} | NaN=0 | Inf=0")
    train_window_pool.append((feat, prices, atrs))

feat_val, price_val, atr_val, _ = featurize(val_slice)
log.info(f"  Val: shape={feat_val.shape}")

feat_matrix = np.concatenate([w[0] for w in train_window_pool], axis=0)
log.info(f"Combined train feat_matrix: {feat_matrix.shape}")


In [ ]:

# Correlation guard
corr = np.corrcoef(feat_matrix.T)
upper = corr[np.triu_indices(len(feat_names), k=1)]
max_corr = np.abs(upper).max()
if max_corr >= 0.80:
    log.warning(f"Feature correlation high: {max_corr:.3f} — consider dropping a feature")
log.info(f"Correlation guard PASSED — all feature pairs |r| < 0.75 (max={max_corr:.3f})")

# Val split: last 30k rows of window[0]
VAL_ROWS = 30_000
feat_val_eval  = train_window_pool[0][0][-VAL_ROWS:].astype(np.float32)
price_val_eval = train_window_pool[0][1][-VAL_ROWS:].astype(np.float32)
atr_val_eval   = train_window_pool[0][2][-VAL_ROWS:].astype(np.float32)
log.info(f"Train pool: {K} windows × {N:,} rows | Val rows: {VAL_ROWS:,} (last 30k of window[0])")

log.info("Adapting Normalization layer on training features ...")
normalizer = tensorflow.keras.layers.Normalization(axis=-1)
normalizer.adapt(feat_matrix)
log.info("Normalization layer adapted.")


In [ ]:

def build_actor_critic(state_dim, n_actions, norm_layer):
    inputs = tensorflow.keras.Input(shape=(state_dim,), name="state", dtype=tf.float32)
    mkt  = inputs[:, :CFG['N_MARKET_FEAT']]
    pos  = inputs[:, CFG['N_MARKET_FEAT']:]
    x    = tensorflow.keras.layers.Concatenate(name="full_state")([norm_layer(mkt), pos])
    x = tensorflow.keras.layers.Dense(128, activation='elu', name='trunk_1', kernel_initializer='he_normal')(x)
    x = tensorflow.keras.layers.LayerNormalization(epsilon=1e-6, name='ln_1')(x)
    x = tensorflow.keras.layers.Dense(64, activation='elu', name='trunk_2', kernel_initializer='he_normal')(x)
    x = tensorflow.keras.layers.LayerNormalization(epsilon=1e-6, name='ln_2')(x)
    a  = tensorflow.keras.layers.Dense(32, activation='elu', name='actor_fc', kernel_initializer='he_normal')(x)
    out_actor  = tensorflow.keras.layers.Dense(n_actions, activation='softmax', name='actor_out', kernel_initializer='orthogonal')(a)
    vc = tensorflow.keras.layers.Dense(32, activation='elu', name='critic_fc', kernel_initializer='he_normal')(x)
    out_critic = tensorflow.keras.layers.Dense(1, activation=None, name='critic_out', kernel_initializer='orthogonal')(vc)
    return tensorflow.keras.Model(inputs=inputs, outputs=[out_actor, out_critic], name="PPO_ActorCritic")

def build_optimizer_and_model():
    m = build_actor_critic(STATE_DIM, N_ACTIONS, normalizer)
    opt = tensorflow.keras.optimizers.Adam(learning_rate=CFG['LR_START'], clipnorm=CFG['MAX_GRAD_NORM'])
    log.info(f"Model built | Parameters: {m.count_params():,}")
    return m, opt

model, optimizer = build_optimizer_and_model()


In [ ]:

class XAUUSDScalpEnv:
    """XAUUSD scalping environment (same as notebook 4)."""
    def __init__(self, window_pool, cfg):
        self.window_pool = [(w[0].astype(np.float32), w[1].astype(np.float32), w[2].astype(np.float32)) for w in window_pool]
        self.cfg         = cfg
        self.features    = self.window_pool[0][0]
        self.prices      = self.window_pool[0][1]
        self.atrs        = self.window_pool[0][2]
        self.n_steps     = len(self.prices)
        self.max_start_row = cfg.get('MAX_START_ROW', None)
        self._mtm_mean = 0.0; self._mtm_var = 1.0; self._mtm_count = 0
        self.reset()

    def reset(self, start=None):
        w = self.window_pool[np.random.randint(len(self.window_pool))]
        self.features = w[0]; self.prices = w[1]; self.atrs = w[2]
        self.n_steps  = len(self.prices)
        warm = 64
        if self.max_start_row is not None:
            max_start = min(self.max_start_row, self.n_steps - warm - 1)
        else:
            max_start = self.n_steps - warm - 1
        self.idx = int(np.clip(start, warm, max_start)) if start is not None else int(np.random.randint(warm, max(warm+1, max_start)))
        self.equity = self.prev_equity = self.peak_equity = float(self.cfg['INIT_EQUITY'])
        self.position = 0; self.entry_price = 0.0; self.bars_held = 0
        self.episode_pnl = 0.0; self.trade_count = 0; self.win_count = 0
        self.prev_price = float(self.prices[self.idx])
        self.prev_trade_equity = float(self.equity)
        self.acc_r_mtm = self.acc_r_cost = self.acc_r_dd = self.n_steps_ep = 0.0
        return self._get_obs()

    def _get_obs(self):
        mkt = self.features[self.idx]
        atr = float(self.atrs[self.idx]) + 1e-8
        unreal_atr = 0.0
        if self.position != 0:
            unreal_atr = float(np.clip((float(self.prices[self.idx]) - self.entry_price) * self.position / atr, -5.0, 5.0))
        dd_pct = max(0.0, (self.peak_equity - self.equity) / (self.peak_equity + 1e-8))
        return np.concatenate([mkt, np.array([float(self.position), self.bars_held / max(self.n_steps,1), unreal_atr, float(np.clip(dd_pct,0,1))], dtype=np.float32)])

    def step(self, action):
        price   = float(self.prices[self.idx])
        atr_val = float(self.atrs[self.idx]) + 1e-8
        lot     = self.cfg['LOT_SIZE']
        desired = 0 if action == 0 else (1 if action == 1 else -1)
        r_cost  = 0.0
        position_changed = (desired != self.position)
        if position_changed:
            raw_cost = self.cfg['SPREAD_PRICE'] / 2.0 + self.cfg['COMMISSION_PRICE']
            r_cost   = float(np.clip(-(raw_cost / atr_val), -2.0, 0.0))
        if self.position != 0:
            self.bars_held += 1
            pnl = (price - self.prev_price) * self.position * lot * 100.0
            self.equity += pnl; self.episode_pnl += pnl
            self.peak_equity = max(self.peak_equity, self.equity)
        if position_changed:
            if self.position != 0:
                if self.equity - self.prev_trade_equity > 0: self.win_count += 1
                if desired != 0: r_cost = float(np.clip(r_cost * 2.0, -2.0, 0.0))
                self.trade_count += 1; self.prev_trade_equity = self.equity
            if desired != 0:
                self.entry_price = price; self.bars_held = 0; self.prev_trade_equity = self.equity
            self.position = desired
        equity_delta = self.equity - self.prev_equity
        r_mtm = float(np.clip(equity_delta / (atr_val * lot * 100.0 + 1e-8), -3.0, 3.0))
        dd_pct = max(0.0, (self.peak_equity - self.equity) / (self.peak_equity + 1e-8))
        r_dd  = float(np.clip(-(dd_pct**2) * self.cfg['LAM_DD'], -2.0, 0.0))
        r_flat = -self.cfg.get('FLAT_PENALTY', 0.0) if desired == 0 else 0.0
        raw_total = r_mtm + r_cost + r_dd + r_flat
        self._mtm_count += 1
        delta = raw_total - self._mtm_mean
        self._mtm_mean += delta / self._mtm_count
        self._mtm_var   = (self._mtm_var * (self._mtm_count-1) + delta*(raw_total - self._mtm_mean)) / max(self._mtm_count, 1)
        _std = math.sqrt(max(self._mtm_var, 1e-6))
        norm_reward = ((raw_total - self._mtm_mean) / _std) if self._mtm_count > 100 else raw_total
        norm_reward = float(np.clip(norm_reward, -self.cfg['REWARD_CLIP'], self.cfg['REWARD_CLIP']))
        self.prev_equity = self.equity
        self.prev_price  = price
        self.idx += 1
        done = (self.idx >= self.n_steps - 1)
        next_obs = self._get_obs() if not done else np.zeros(STATE_DIM, dtype=np.float32)
        info = {"r_mtm": r_mtm, "r_cost": r_cost, "r_dd": r_dd,
                "equity": self.equity, "dd_pct": dd_pct, "position": self.position,
                "next_obs": next_obs.copy()}
        return next_obs, norm_reward, done, info


class XAUUSDGymEnv:
    """Gymnasium-compatible wrapper (reset→(obs,info), step→(obs,r,term,trunc,info))."""
    def __init__(self, window_pool, cfg):
        self._env = XAUUSDScalpEnv(window_pool, cfg)

    def reset(self, seed=None, options=None):
        obs = self._env.reset()
        return obs, {}

    def step(self, action):
        next_obs, reward, done, info = self._env.step(action)
        return next_obs, reward, bool(done), False, info

    def close(self):
        pass

    @property
    def inner(self):
        return self._env

log.info("Environments defined.")


In [ ]:

@tf.function(reduce_retracing=True)
def _compute_gae(rewards, values, dones, last_val, gamma, lam):
    n          = tf.shape(rewards)[0]
    advantages = tf.TensorArray(dtype=tf.float32, size=n, dynamic_size=False)
    gae        = tf.constant(0.0)
    for t in tf.range(n - 1, -1, -1):
        next_v   = tf.cond(t == n - 1, lambda: last_val, lambda: values[t + 1])
        not_done = 1.0 - dones[t]
        delta    = rewards[t] + gamma * next_v * not_done - values[t]
        gae      = delta + gamma * lam * not_done * tf.cast(gae, tf.float32)
        advantages = advantages.write(t, gae)
    adv  = advantages.stack()
    rets = tf.cast(adv, tf.float32) + values
    adv_n = (tf.cast(adv, tf.float32) - tf.reduce_mean(adv)) / (tf.math.reduce_std(adv) + 1e-8)
    return adv_n, rets


@tf.function(reduce_retracing=True)
def _ppo_step(m, opt, obs_b, act_b, old_lp_b, old_val_b, adv_b, ret_b,
              clip_eps, entropy_coeff, entropy_target, value_coeff):
    with tf.GradientTape() as tape:
        probs, values = m(obs_b, training=True)
        values = tf.squeeze(values, -1)
        log_probs = tf.math.log(tf.clip_by_value(probs, 1e-8, 1.0))
        act_oh    = tf.one_hot(tf.cast(act_b, tf.int32), N_ACTIONS)
        log_pi_a  = tf.reduce_sum(log_probs * act_oh, axis=-1)
        ratio     = tf.exp(log_pi_a - old_lp_b)
        clip_adv  = tf.clip_by_value(ratio, 1-clip_eps, 1+clip_eps) * adv_b
        actor_loss= -tf.reduce_mean(tf.minimum(ratio * adv_b, clip_adv))
        v_clipped = old_val_b + tf.clip_by_value(values - old_val_b, -clip_eps, clip_eps)
        value_loss= tf.reduce_mean(tf.maximum(tf.square(ret_b - values), tf.square(ret_b - v_clipped)))
        entropy   = tf.reduce_mean(-tf.reduce_sum(probs * log_probs, axis=-1))
        ent_deficit = tf.maximum(0.0, entropy_target - entropy)
        total_loss  = actor_loss + value_coeff * value_loss - entropy_coeff * entropy + entropy_coeff * ent_deficit
    grads = tape.gradient(total_loss, m.trainable_variables)
    grads = [tf.clip_by_norm(g, 0.5) if g is not None else g for g in grads]
    opt.apply_gradients(zip(grads, m.trainable_variables))
    kl        = tf.reduce_mean(old_lp_b - log_pi_a)
    clip_frac = tf.reduce_mean(tf.cast(tf.abs(ratio - 1.0) > clip_eps, tf.float32))
    return total_loss, actor_loss, value_loss, entropy, kl, clip_frac


class XAUUSDPPOAgent(BaseAgent):
    """
    PPO agent wrapping the shared actor-critic model.
    Implements the 6-method BaseAgent contract for tensor-optix.
    """
    def __init__(self, model, optimizer, hyperparams: HyperparamSet):
        self._model     = model
        self._optimizer = optimizer
        self._hp        = hyperparams.copy()
        # Internal buffers — filled during act(), consumed in learn()
        self._log_probs: list = []
        self._values:    list = []

    # ── act ───────────────────────────────────────────────────────────────────
    def act(self, observation) -> int:
        obs_tf = tf.expand_dims(tf.cast(observation, tf.float32), 0)
        probs, value = self._model(obs_tf, training=False)
        dist   = tf.squeeze(probs, 0)
        action = int(tf.squeeze(tf.cast(
            tf.random.categorical(tf.math.log(tf.clip_by_value(dist[tf.newaxis], 1e-8, 1.0)), 1),
            tf.int32)).numpy())
        log_p  = float(tf.math.log(tf.clip_by_value(dist[action], 1e-8, 1.0)).numpy())
        val    = float(tf.squeeze(value).numpy())
        self._log_probs.append(log_p)
        self._values.append(val)
        return action

    # ── learn ─────────────────────────────────────────────────────────────────
    def learn(self, episode_data: EpisodeData) -> dict:
        hp = self._hp.params
        gamma     = float(hp.get('gamma',          CFG['GAMMA']))
        lam       = float(hp.get('gae_lambda',     CFG['GAE_LAMBDA']))
        clip_eps  = float(hp.get('clip_ratio',     CFG['CLIP_EPS']))
        ent_coeff = float(hp.get('entropy_coeff',  CFG['ENTROPY_COEFF']))
        ent_target= float(hp.get('entropy_target', CFG['ENTROPY_TARGET']))
        val_coeff = float(hp.get('value_coeff',    CFG['VALUE_COEFF']))
        n_epochs  = int(hp.get('n_epochs',         CFG['N_EPOCHS']))
        batch_sz  = int(hp.get('batch_size',       CFG['MINI_BATCH_SIZE']))
        target_kl = float(hp.get('target_kl',      CFG['TARGET_KL']))

        T = len(episode_data.rewards)
        obs_arr    = tf.cast(episode_data.observations[:T], tf.float32)
        act_arr    = tf.cast(episode_data.actions[:T], tf.int32)
        rew_tf     = tf.cast(episode_data.rewards[:T], tf.float32)
        done_tf    = tf.cast([float(t or tr) for t, tr in
                              zip(episode_data.terminated[:T], episode_data.truncated[:T])], tf.float32)
        old_lp_tf  = tf.cast(self._log_probs[:T], tf.float32)
        val_tf     = tf.cast(self._values[:T],    tf.float32)

        # Bootstrap last value from next_obs stored in info
        last_info = episode_data.infos[-1] if episode_data.infos else {}
        last_next = last_info.get('next_obs', None)
        if last_next is not None and not (episode_data.terminated[-1] or episode_data.truncated[-1]):
            _, lv = self._model(tf.expand_dims(tf.cast(last_next, tf.float32), 0), training=False)
            last_val = tf.squeeze(lv)
        else:
            last_val = tf.constant(0.0)

        adv_tf, ret_tf = _compute_gae(rew_tf, val_tf, done_tf, last_val, gamma, lam)

        # Multi-epoch mini-batch PPO
        total_losses = []; entropies = []; kls = []; clip_fracs = []
        idx = np.arange(T)
        for epoch in range(n_epochs):
            np.random.shuffle(idx)
            kl_early = False
            for start in range(0, T, batch_sz):
                b = idx[start: start + batch_sz]
                tl, al, vl, ent, kl, cf = _ppo_step(
                    self._model, self._optimizer,
                    tf.gather(obs_arr, b), tf.gather(act_arr, b),
                    tf.gather(old_lp_tf, b), tf.gather(val_tf, b),
                    tf.gather(adv_tf, b), tf.gather(ret_tf, b),
                    tf.constant(clip_eps, tf.float32), tf.constant(ent_coeff, tf.float32),
                    tf.constant(ent_target, tf.float32), tf.constant(val_coeff, tf.float32),
                )
                total_losses.append(float(tl)); entropies.append(float(ent))
                kls.append(float(kl)); clip_fracs.append(float(cf))
                if float(kl) > target_kl:
                    kl_early = True; break
            if kl_early:
                break

        # Derive action distribution for logging
        probs_all, _ = self._model(obs_arr[:256], training=False)
        probs_mean   = tf.reduce_mean(probs_all, 0).numpy()

        # Clear buffers
        self._log_probs.clear()
        self._values.clear()

        return {
            "loss":       float(np.mean(total_losses)),
            "entropy":    float(np.mean(entropies)),
            "kl":         float(np.mean(kls)),
            "clip_frac":  float(np.mean(clip_fracs)),
            "p_flat":     float(probs_mean[0]),
            "p_long":     float(probs_mean[1]),
            "p_short":    float(probs_mean[2]),
        }

    # ── hyperparams ──────────────────────────────────────────────────────────
    def get_hyperparams(self) -> HyperparamSet:
        self._hp.params['learning_rate'] = float(self._optimizer.learning_rate.numpy()
            if hasattr(self._optimizer.learning_rate, 'numpy') else self._optimizer.learning_rate)
        return self._hp.copy()

    def set_hyperparams(self, hp: HyperparamSet):
        self._hp = hp.copy()
        if 'learning_rate' in hp.params:
            self._optimizer.learning_rate.assign(float(hp.params['learning_rate']))

    # ── weights ──────────────────────────────────────────────────────────────
    def save_weights(self, path: str):
        os.makedirs(path, exist_ok=True)
        self._model.save(os.path.join(path, "model.keras"))

    def load_weights(self, path: str):
        loaded = tf.keras.models.load_model(os.path.join(path, "model.keras"))
        for v, lv in zip(self._model.trainable_variables, loaded.trainable_variables):
            v.assign(lv)

log.info("XAUUSDPPOAgent defined.")


In [ ]:

class XAUUSDEvaluator(BaseEvaluator):
    """
    Primary score = annualised Sharpe from equity curve.
    Falls back to reward-based Sharpe if equity not in infos.
    """
    def score(self, episode_data: EpisodeData, train_diagnostics: dict) -> EvalMetrics:
        equities = [info.get('equity', None) for info in episode_data.infos]
        equities = [e for e in equities if e is not None]

        if len(equities) > 1:
            eq = np.array(equities, dtype=np.float32)
            returns = np.diff(eq) / (eq[:-1] + 1e-8)
            sharpe  = float(returns.mean() / (returns.std() + 1e-8)) * np.sqrt(252 * 1440)
            pnl     = float(eq[-1] - eq[0])
            peak    = float(eq.max())
            trough  = float(eq[eq.argmax():].min()) if eq.argmax() < len(eq)-1 else float(eq[-1])
            max_dd  = float((peak - trough) / (peak + 1e-8))
        else:
            rewards = np.array(episode_data.rewards, dtype=np.float32)
            sharpe  = float(rewards.mean() / (rewards.std() + 1e-8))
            pnl     = float(rewards.sum())
            max_dd  = 0.0

        positions = [info.get('position', 0) for info in episode_data.infos]
        n_long  = sum(1 for p in positions if p == 1)
        n_short = sum(1 for p in positions if p == -1)
        n_flat  = sum(1 for p in positions if p == 0)

        metrics = {
            "sharpe":  sharpe,
            "pnl":     pnl,
            "max_dd":  max_dd,
            "n_long":  float(n_long),
            "n_short": float(n_short),
            "n_flat":  float(n_flat),
            **{k: float(v) for k, v in train_diagnostics.items() if isinstance(v, (int, float))},
        }
        return EvalMetrics(primary_score=sharpe, metrics=metrics, episode_id=episode_data.episode_id)

    def combine(self, train: EvalMetrics, val: EvalMetrics) -> EvalMetrics:
        """Val-primary combined score — val drives all adaptation decisions."""
        gap = train.primary_score - val.primary_score
        return EvalMetrics(
            primary_score=val.primary_score,
            metrics={
                **{f"train_{k}": v for k, v in train.metrics.items()},
                **{f"val_{k}":   v for k, v in val.metrics.items()},
                "train_score":        train.primary_score,
                "val_score":          val.primary_score,
                "generalization_gap": gap,
            },
            episode_id=train.episode_id,
        )

log.info("XAUUSDEvaluator defined.")


In [ ]:

# Train env: 20 windows, capped starts
train_gym = XAUUSDGymEnv(train_window_pool, CFG)

# Val env: 30k rows, no start cap
_val_cfg = {**CFG, "MAX_START_ROW": None}
val_gym  = XAUUSDGymEnv([(feat_val_eval, price_val_eval, atr_val_eval)], _val_cfg)

# Initial hyperparams — PBT will tune entropy_coeff, clip_ratio, learning_rate
initial_hp = HyperparamSet(
    params={
        "learning_rate":  CFG['LR_START'],
        "clip_ratio":     CFG['CLIP_EPS'],
        "entropy_coeff":  CFG['ENTROPY_COEFF'],
        "entropy_target": CFG['ENTROPY_TARGET'],
        "gamma":          CFG['GAMMA'],
        "gae_lambda":     CFG['GAE_LAMBDA'],
        "value_coeff":    CFG['VALUE_COEFF'],
        "n_epochs":       CFG['N_EPOCHS'],
        "batch_size":     CFG['MINI_BATCH_SIZE'],
        "target_kl":      CFG['TARGET_KL'],
    },
    episode_id=0,
)

agent = XAUUSDPPOAgent(model, optimizer, initial_hp)

train_pipeline = BatchPipeline(env=train_gym, agent=agent, window_size=CFG['N_ROLLOUT_STEPS'])
val_pipeline   = BatchPipeline(env=val_gym,   agent=agent, window_size=CFG['N_ROLLOUT_STEPS'])

log.info(f"Pipelines ready | train={K} windows × {N:,} rows | val={VAL_ROWS:,} rows")


In [ ]:

class XAUUSDLogger(LoopCallback):
    """Logs every eval and key loop events to training.log + JSONL."""

    def on_episode_end(self, episode_id: int, eval_metrics) -> None:
        score = eval_metrics.primary_score if eval_metrics else 0.0
        m     = eval_metrics.metrics       if eval_metrics else {}
        ent   = m.get('train_entropy', m.get('entropy', 0.0))
        loss  = m.get('train_loss',    m.get('loss',    0.0))
        pf    = m.get('train_p_flat',  m.get('p_flat',  0.0))
        pl    = m.get('train_p_long',  m.get('p_long',  0.0))
        ps    = m.get('train_p_short', m.get('p_short', 0.0))
        eq    = m.get('train_pnl',     m.get('pnl',     0.0))
        log.info(
            f"[W{episode_id:04d}] score={score:.4f} | loss={loss:.5f} | entropy={ent:.5f} | "
            f"actions=[F:{pf:.2f} L:{pl:.2f} S:{ps:.2f}] | pnl=${eq:.2f}"
        )
        jlog({"event": "window", "episode_id": episode_id, "score": score,
              "entropy": ent, "loss": loss})

    def on_improvement(self, snapshot):
        log.info(f"  [BEST] New best score={snapshot.eval_metrics.primary_score:.4f} → {snapshot.weights_path}")
        jlog({"event": "best", "score": snapshot.eval_metrics.primary_score})

    def on_plateau(self, window_id: int):
        log.info(f"  [PLATEAU] window={window_id}")

    def on_dormant(self, window_id: int):
        log.info(f"  [DORMANT] Converged at window={window_id}")
        jlog({"event": "dormant", "window_id": window_id})

    def on_hyperparam_update(self, old_params: dict, new_params: dict) -> None:
        changed = {k: new_params[k] for k in new_params
                   if abs(new_params.get(k, 0) - old_params.get(k, 0)) > 1e-9}
        if changed:
            log.info(f"  [PBT] Hyperparams updated: {changed}")
            jlog({"event": "pbt_update", "changed": changed})

log.info("XAUUSDLogger callback defined.")


In [ ]:

registry = CheckpointRegistry("output/to_checkpoints", max_snapshots=10)

pm = PolicyManager(registry, max_spawns=5, max_ensemble_size=3)

def make_agent():
    """Factory for spawning fresh policy variants."""
    m, opt = build_optimizer_and_model()
    return XAUUSDPPOAgent(m, opt, initial_hp)

meta = MetaController(
    gap_threshold=0.5,          # normalized overfitting gap above this → PRUNE
    gap_slope_threshold=0.03,   # gap widening faster than this → PRUNE
    improvement_threshold=0.02, # normalized val slope below this → SPAWN
)

pm_callback = pm.as_callback(
    agent,
    agent_factory=make_agent,
    meta_controller=meta,
)


# PBT optimizer — tunes entropy_coeff, clip_ratio, learning_rate
pbt = PBTOptimizer(
    param_bounds={
        "entropy_coeff":  (0.01, 0.15),
        "clip_ratio":     (0.10, 0.35),
        "learning_rate":  (1e-5, 5e-4),
    },
    log_scale_params={"learning_rate"},   # multiplicative noise for LR
    history_size=30,
    explore_scale=0.10,
    exploit_scale=0.05,
)

log.info("PolicyManager + PBTOptimizer configured.")


In [ ]:

opt = RLOptimizer(
    agent          = agent,
    pipeline       = train_pipeline,
    val_pipeline   = val_pipeline,
    evaluator      = XAUUSDEvaluator(),
    optimizer      = pbt,
    checkpoint_dir = "output/to_checkpoints",
    max_snapshots  = 10,
    rollback_on_degradation = True,
    improvement_margin      = 0.01,
    base_interval           = 1,
    backoff_factor          = 2.0,
    max_interval_episodes   = 50,
    plateau_threshold       = 5,
    dormant_threshold       = 20,
    degradation_threshold   = 0.95,
    min_degradation_drop    = 2.0,
    callbacks      = [XAUUSDLogger(), pm_callback],
)

pm_callback.set_stop_fn(opt.stop)

log.info("=" * 72)
log.info("XAUUSD PPO — tensor-optix autonomous training")
log.info(f"Rollout window: {CFG['N_ROLLOUT_STEPS']} steps | "
         f"PBT tuning: entropy_coeff, clip_ratio, lr | "
         f"Max spawns: 5")
pm_callback.set_stop_fn(opt.stop)

log.info("=" * 72)

opt.run()

pm_callback.set_stop_fn(opt.stop)

log.info("=" * 72)
log.info(f"Training complete | Best score: {opt.best_snapshot.eval_metrics.primary_score:.4f}")
log.info(f"Ensemble status: {json.dumps(pm.status(), indent=2)}")
pm_callback.set_stop_fn(opt.stop)

log.info("=" * 72)


In [ ]:

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

log.info("Loading best model for backtest ...")

# Load best weights back into model
if opt.best_snapshot:
    agent.load_weights(opt.best_snapshot.weights_path)
    log.info(f"  Best score: {opt.best_snapshot.eval_metrics.primary_score:.4f}")

# Run full deterministic episode on window[0] from start=64
bt_env = XAUUSDScalpEnv([(train_window_pool[0][0], train_window_pool[0][1], train_window_pool[0][2])],
                         {**CFG, "MAX_START_ROW": None})
obs    = bt_env.reset(start=64)
equities, actions_log = [bt_env.equity], []
done   = False
while not done:
    obs_tf = tf.expand_dims(tf.cast(obs, tf.float32), 0)
    probs, _ = model(obs_tf, training=False)
    action   = int(tf.argmax(tf.squeeze(probs)).numpy())
    obs, _, done, info = bt_env.step(action)
    equities.append(info['equity'])
    actions_log.append(info['position'])

eq = np.array(equities)
rets = np.diff(eq) / (eq[:-1] + 1e-8)
sharpe = float(rets.mean() / (rets.std() + 1e-8)) * np.sqrt(252 * 1440)
total_ret = (eq[-1] / eq[0] - 1) * 100
peak = eq.max(); trough = eq[eq.argmax():].min()
max_dd = (peak - trough) / (peak + 1e-8) * 100
trade_changes = sum(1 for i in range(1, len(actions_log)) if actions_log[i] != actions_log[i-1])

log.info(f"Backtest | Return={total_ret:.2f}% | Sharpe={sharpe:.2f} | MaxDD={max_dd:.2f}% | Trades={trade_changes}")

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(eq, color='steelblue', linewidth=0.8)
axes[0].axhline(eq[0], color='gray', linestyle='--', linewidth=0.5)
axes[0].set_ylabel("Equity ($)"); axes[0].set_title(
    f"Backtest — Return {total_ret:.2f}% | Sharpe {sharpe:.2f} | MaxDD {max_dd:.2f}%")
axes[1].plot(actions_log, color='darkorange', linewidth=0.5, alpha=0.7)
axes[1].set_yticks([-1,0,1]); axes[1].set_yticklabels(['SHORT','FLAT','LONG'])
axes[1].set_ylabel("Position"); axes[1].set_xlabel("Step")
plt.tight_layout()
plt.savefig("output/to_backtest_equity_curve.png", dpi=120)
plt.show()
log.info("Backtest chart saved → output/to_backtest_equity_curve.png")
